# DJR WGS sex versus field sex

Compare WGS/Y-based sex assignments with field metadata. If WGS sex is missing, retain field sex.

In [ ]:
library(data.table)

wgs_file <- "/scratch/ztobias/BC_stickle_expanded_WGS/metadata/derived/wgs_sex_assignment_reconciliation_DJR.tsv"
dna24_file <- "/scratch/ztobias/Gacu_BC_txmics/metadata/2024_BC_DNA_extractions.csv"
meta25_file <- "/scratch/ztobias/Gacu_BC_txmics/metadata/BC Spring 2025 Collections and Molecular Metadata Sheet 1.csv"

wgs <- fread(wgs_file)
dna24 <- fread(dna24_file, fill = TRUE)
meta25 <- fread(meta25_file, fill = TRUE)

norm_sex <- function(x) {
  x <- tolower(trimws(as.character(x)))
  fifelse(grepl("^f|female", x), "female",
    fifelse(grepl("^m|male", x), "male", NA_character_))
}

wgs <- wgs[grepl("^(2024|2025)_DJR_", sample)]
wgs[, wgs_sex := norm_sex(assigned_sex_y_djr)]
wgs <- wgs[, .(sample, wgs_sex, y_missing_proportion,
  assignment_margin_djr, assignment_confident_djr)]

field24 <- dna24[grepl("^(2024|2025)_DJR_", sample), .(
  sample, field_sex = norm_sex(sex),
  field_notes = as.character(get("extr notes"))
)]

setnames(meta25, names(meta25)[1], "sample")
sex_col <- names(meta25)[7]          # Sex
eggs_col <- names(meta25)[8]         # Eggs?
notes_col <- names(meta25)[10]       # Field notes
stained_sex_col <- names(meta25)[14] # stained sex check
field25 <- meta25[grepl("^(2024|2025)_DJR_", sample), .(
  sample, field_sex = norm_sex(get(sex_col)),
  field_notes = paste(get(notes_col), get(eggs_col), get(stained_sex_col), sep = " | ")
)]

field <- rbindlist(list(field24, field25), fill = TRUE)
field <- field[!duplicated(sample)]

sex_compare <- merge(field, wgs, by = "sample", all = TRUE)
sex_compare[, mismatch := !is.na(field_sex) & !is.na(wgs_sex) & field_sex != wgs_sex]
sex_compare[, final_sex := fifelse(!is.na(wgs_sex), wgs_sex, field_sex)]
sex_compare[, final_sex_source := fifelse(!is.na(wgs_sex), "WGS",
  fifelse(!is.na(field_sex), "field", NA_character_))]

cat("Mismatches:\n")
print(sex_compare[mismatch == TRUE])
cat("\nField-sex fallback samples:\n")
print(sex_compare[is.na(wgs_sex) & !is.na(field_sex)])

In [ ]:
out_dir <- "/scratch/ztobias/Gacu_BC_txmics/metadata/derived"
fwrite(sex_compare, file.path(out_dir, "DJR_sex_field_comparison.tsv"), sep = "\t")
fwrite(sex_compare[mismatch == TRUE], file.path(out_dir, "DJR_sex_mismatches.tsv"), sep = "\t")

sex_compare[, .(n = .N), by = .(final_sex_source, final_sex)]

In [ ]:
# BRB-seq samples: retain only fish with a gill and/or liver barcode in the manifest.
brb <- fread("/scratch/ztobias/Gacu_BC_txmics/metadata/brb_sample_barcodes.tsv")
brb <- brb[grepl("^(2024|2025)_DJR_", sample)]
brb[, tissue := tolower(tissue)]
brb_presence <- unique(brb[, .(sample, tissue)])
brb_presence[, present := TRUE]
brb_presence <- dcast(brb_presence, sample ~ tissue, value.var = "present", fill = FALSE)
if (!"gill" %in% names(brb_presence)) brb_presence[, gill := FALSE]
if (!"liver" %in% names(brb_presence)) brb_presence[, liver := FALSE]
setnames(brb_presence, c("gill", "liver"), c("brb_gill", "brb_liver"))

brb_sex <- merge(brb_presence, sex_compare, by = "sample", all.x = TRUE)
brb_sex[, brb_tissues := fifelse(brb_gill & brb_liver, "gill+liver",
  fifelse(brb_gill, "gill", "liver"))]
brb_sex <- brb_sex[order(sample)]
brb_sex <- brb_sex[, .(sample, brb_tissues, brb_gill, brb_liver,
  field_sex, wgs_sex, final_sex, final_sex_source, mismatch,
  y_missing_proportion, assignment_confident_djr, field_notes)]

cat("BRB-seq samples by tissue:\n")
print(brb_sex[, .(n = .N), by = brb_tissues])
cat("BRB-seq samples without WGS sex:\n")
print(brb_sex[is.na(wgs_sex), .(sample, brb_tissues, field_sex, final_sex, field_notes)])

fwrite(brb_sex, file.path(out_dir, "DJR_sex_BRB_samples.tsv"), sep = "\t")
brb_sex